In [1]:
"""
%pip install qiskit-nature==0.7.2
%pip install qiskit-aer==0.17.2
%pip install pyswarms==1.3.0
%pip install pylatexenc==2.10
%pip install qiskit==2.3.0
%pip install numpy==2.4.2
%pip install pyscf==2.12.1
"""

'\n%pip install qiskit-nature==0.7.2\n%pip install qiskit-aer==0.17.2\n%pip install pyswarms==1.3.0\n%pip install pylatexenc==2.10\n%pip install qiskit==2.3.0\n%pip install numpy==2.4.2\n%pip install pyscf==2.12.1\n'

In [2]:
import os
import time
import json
import logging
import numpy as np
import pandas as pd
from scipy.stats import qmc
import pyswarms.backend as P
from qiskit_aer import AerSimulator
from joblib import Parallel, delayed
from scipy.optimize import OptimizeResult
from pyswarms.backend.topology import Star
from qiskit_nature.units import DistanceUnit
from qiskit.primitives import StatevectorEstimator
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_algorithms import MinimumEigensolverResult
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

2026-07-27 22:19:49,693 - qiskit.passmanager.base_tasks - INFO - Pass: UnrollCustomDefinitions - 0.10180 (ms)
2026-07-27 22:19:49,694 - qiskit.passmanager.base_tasks - INFO - Pass: BasisTranslator - 0.02050 (ms)


In [3]:
logging.getLogger('qiskit_nature').setLevel(logging.WARNING)

# 1°: Building the Molecular Problem


In [4]:
# Creating the molecular geometry of BeH2 with STO-3G as minimal basis set and
# H-Be distance of 1.326 angstrom and 180 degrees,

driver = PySCFDriver(
        atom="""H -1.326, 0.0, 0.0
                Be 0.0, 0.0, 0.0
                H 1.326, 0.0, 0.0
             """,
        basis='sto3g',
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM
)

molecule_problem = driver.run()

In [5]:
# The original space has approximately 6 electrons, 6 space orbitals, and 12
# spin orbitals.
# Here we limit the active space to only 4 electrons and 3 space orbitals and
# work with the reduced molecule_problem

active_space_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=3)

reduced_molecule_problem = active_space_transformer.transform(molecule_problem)

# 2°: The Hamiltonian in Terms of Qubits

In [6]:
# Here we calculate the Hamiltonian of the second quantization after reduction
# with CAS.
second_q_hamiltonian = reduced_molecule_problem.second_q_ops()[0]

In [7]:
# Here we use the Jordan Wigner mapping
jordan_wigner_mapper = JordanWignerMapper()

qubit_op = jordan_wigner_mapper.map(second_q_hamiltonian)

In [8]:
# Finally, we obtain the number of qubits and operator in terms of Pauli matrices.
num_qubits = qubit_op.num_qubits
print(f"Number of qubits = {num_qubits}")
print(f"Hamiltonian: {qubit_op}")

Number of qubits = 6
Hamiltonian: SparsePauliOp(['IIIIII', 'IIIIIZ', 'IIIIZI', 'IIIZII', 'IIZIII', 'IZIIII', 'ZIIIII', 'IIIIZZ', 'IIIZIZ', 'IIZIIZ', 'IZIIIZ', 'ZIIIIZ', 'IYYIYY', 'IXXIYY', 'IYYIXX', 'IXXIXX', 'YZYYZY', 'XZXYZY', 'YZYXZX', 'XZXXZX', 'IIIZZI', 'IIZIZI', 'IZIIZI', 'ZIIIZI', 'YYIYYI', 'XXIYYI', 'YYIXXI', 'XXIXXI', 'IIZZII', 'IZIZII', 'ZIIZII', 'IZZIII', 'ZIZIII', 'ZZIIII'],
              coeffs=[-2.86740153+0.j,  0.32161792+0.j,  0.31034865+0.j,  0.12889394+0.j,
  0.32161792+0.j,  0.31034865+0.j,  0.12889394+0.j,  0.06196671+0.j,
  0.08001574+0.j,  0.09975793+0.j,  0.10309655+0.j,  0.09238729+0.j,
  0.04112984+0.j,  0.04112984+0.j,  0.04112984+0.j,  0.04112984+0.j,
  0.01237154+0.j,  0.01237154+0.j,  0.01237154+0.j,  0.01237154+0.j,
  0.08557179+0.j,  0.10309655+0.j,  0.10888574+0.j,  0.08924797+0.j,
  0.00367617+0.j,  0.00367617+0.j,  0.00367617+0.j,  0.00367617+0.j,
  0.09238729+0.j,  0.08924797+0.j,  0.11246476+0.j,  0.06196671+0.j,
  0.08001574+0.j,  0.08557179+0.j])


# 3°: Ansatz Circuit Construction


In [9]:
# Here we construct the HF state within the CAS and JW mapping.
hf_initial_state = HartreeFock(
      num_particles=reduced_molecule_problem.num_particles,
      num_spatial_orbitals=reduced_molecule_problem.num_spatial_orbitals,
      qubit_mapper=jordan_wigner_mapper
)

In [10]:
# Here we build the ansatz
ansatz = UCCSD(
          reduced_molecule_problem.num_spatial_orbitals,
          reduced_molecule_problem.num_particles,
          initial_state=hf_initial_state,
          qubit_mapper=jordan_wigner_mapper
)

# 4°: Transpilation and Simulator Settings

In [11]:
# This small function was created to generate the ansatz and observables in
# terms of Instruction Set Architecture (ISA) operators.
def transpile_to_isa(backend):
  target = backend.target
  pm = generate_preset_pass_manager(target=target)
  ansatz_isa = pm.run(ansatz)
  isa_observables = qubit_op.apply_layout(ansatz_isa.layout)
  return ansatz_isa, isa_observables

# 5°: Measuring Eigenvalues and Energies

In [12]:
# Here we add the energy of active space to the frozen energies of the core
# plus nuclear repulsion.
def interpret_exp_val(exp_val, problem):
    sol = MinimumEigensolverResult()
    sol.eigenvalue = np.real(exp_val)
    return problem.interpret(sol).total_energies[0]

# 6°: Creating a custom Global Best PSO loop

The custom GPSO class will allow us to directly implement the GPSO optimizer as if we were implementing a class from qiskit algorithms/machine learning or from scipy itself.

In [13]:
class QPSO:
    def __init__(self, maxiter: int, n_particles: int, dimensions: int, options: dict = None, bounds=(-np.pi, np.pi), callback=None, seed: int = None):
        self.maxiters = maxiter
        self.n_particles = n_particles
        self.dimensions = dimensions
        self.callback = callback
        self.bounds = bounds # Assigning bounds for boundary handling
        self.seed = seed

        # Initialize the local random number generator (RNG)
        self.rng = np.random.default_rng(seed)

        # QPSO contraction-expansion parameter (alpha or beta).
        # Typically decays linearly from alpha_max to alpha_min.
        if options is None:
            options = {}
        self.alpha_max = options.get('alpha_max', 1.0)
        self.alpha_min = options.get('alpha_min', 0.5)

    def minimize(self, fun, x0=None):
        nit = 0
        nfev = 0
        lower_b, upper_b = self.bounds

        # 1. HALTON SEQUENCE INITIALIZATION
        # Initialize the Halton sampler for the problem dimensions
        # We pass the local RNG to ensure the sequence is reproducible
        sampler = qmc.Halton(d=self.dimensions, scramble=True, seed=self.rng)
        sample = sampler.random(n=self.n_particles)

        # Scale the sample from [0, 1] to the specified bounds
        position = qmc.scale(sample, lower_b, upper_b)

        # 2. ANCHORING x0 (e.g., Hartree-Fock state)
        if x0 is not None:
            position[0] = np.array(x0)

        # Initialize swarm memories (Pbest and Gbest)
        pbest_pos = np.copy(position)
        pbest_cost = np.full(self.n_particles, np.inf, dtype=float)

        gbest_pos = np.zeros(self.dimensions)
        gbest_cost = np.inf

        # 3. MAIN QPSO OPTIMIZATION LOOP
        for t in range(self.maxiters):

            # --- Energy Evaluation ---
            for i in range(self.n_particles):
                cost = fun(position[i])
                nfev += 1

                # Update Personal Best (Pbest)
                if cost < pbest_cost[i]:
                    pbest_cost[i] = cost
                    pbest_pos[i] = np.copy(position[i])

                    # Update Global Best (Gbest)
                    if cost < gbest_cost:
                        gbest_cost = cost
                        gbest_pos = np.copy(position[i])

            # --- Callback Execution ---
            if self.callback is not None:
                self.callback(gbest_pos, gbest_cost)

            # --- Topological Updates (Quantum Wavefunction Dynamics) ---
            # 1. Calculate mbest (Mean of all personal bests)
            mbest = np.mean(pbest_pos, axis=0)

            # 2. Dynamic alpha calculation (Linear decay)
            alpha = self.alpha_max - (self.alpha_max - self.alpha_min) * (t / self.maxiters)

            # 3. Wavefunction collapse for each particle
            for i in range(self.n_particles):
                # phi is a random vector between 0 and 1 generated by the local RNG
                phi = self.rng.random(self.dimensions)

                # Local attractor point (Intersection between pbest and gbest)
                p_attractor = phi * pbest_pos[i] + (1 - phi) * gbest_pos

                # u determines the quantum jump length (Avoid 0 to prevent log errors)
                u = self.rng.random(self.dimensions)
                u = np.clip(u, 1e-10, 1.0)

                # Potential well characteristic length (L)
                L = alpha * np.abs(mbest - position[i])

                # Directional choice (Stochastic sign assignment) using local RNG
                sign = self.rng.choice([-1, 1], size=self.dimensions)

                # Update position and apply Boundary Handling (Clamping)
                new_pos = p_attractor + sign * L * np.log(1 / u)
                position[i] = np.clip(new_pos, lower_b, upper_b)

            nit += 1

        # Return the final result in a SciPy-compatible format
        result = OptimizeResult(
            fun=float(gbest_cost),
            x=np.array(gbest_pos, dtype=float),
            nit=nit,
            nfev=nfev,
            success=True,
            message="QPSO with Halton Initialization terminated successfully."
        )

        return result

# 7°: Configuring Parallel Execution Function

In [14]:
def parallel_optimization(run_idx: int, maxiter: int, qpso_seed:int):

  # Initialization of the Aer simulator
  backend = AerSimulator(method='statevector', device="CPU")

  # Ansatz transpilation and observables for the ISA
  ansatz_isa, isa_observables = transpile_to_isa(backend)

  # Declaring and configuring the Estimator for calculating expected values.
  estimator = StatevectorEstimator()

  # The energies and parameters per iteration will be stored here.
  energy_data = []
  params_data = []

  # Initializing parameters as all at 0. This means we will start in the HF state.
  initial_params = np.zeros(ansatz_isa.num_parameters)

  # Callback function declaration
  def optimizer_callback(params, value):
    energy_data.append(value)
    params_data.append(params.tolist())

  # Statement of the cost function
  def energy_cost_function(params):
    estimator_job = estimator.run([(ansatz_isa, isa_observables, params)])
    estimator_exp_val = estimator_job.result()[0].data.evs
    return float(estimator_exp_val)

  # Optimizer instance with settings
  optimizer = QPSO(
              maxiter=maxiter,
              n_particles=20,
              dimensions=ansatz_isa.num_parameters,
              bounds=(-np.pi, np.pi),
              options={'alpha_max': 1.0, 'alpha_min': 0.5},
              callback=optimizer_callback,
              seed=qpso_seed
            )

  # The optimization process takes place here.
  t0 = time.perf_counter()
  result = optimizer.minimize(fun=energy_cost_function, x0=initial_params)
  t1 = time.perf_counter()

  return {
        "optimizer": "QPSO",
        "run": run_idx,
        "qpso_seed":qpso_seed,
        "steps_requested": maxiter,
        "steps_done": result.nit,
        "cost_function_evaluation": result.nfev,
        "energies_trajectory_len": len(energy_data),
        "execution_time": t1 - t0,
        "final_energy": float(result.fun),
        "optimal_params": result.x.tolist(),
        "energies_trajectory": energy_data,
        "params_trajectory": params_data,
  }

# 8°: Running VQE in parallel on CPU cores

In [15]:
maxiter = 1000
runs = 50
qpso_seed = 212

In [16]:
n_jobs = os.cpu_count()
print(f"Running {runs} QPSO runs in parallel (CPU) | n_jobs={n_jobs} | iteration={maxiter}")

Running 50 QPSO runs in parallel (CPU) | n_jobs=44 | iteration=1000


In [17]:
results = Parallel(n_jobs=n_jobs, backend="loky", verbose=10)(
    delayed(parallel_optimization)(
        run_idx=i,
        maxiter=maxiter,
        qpso_seed=qpso_seed+i
    )
    for i in range(runs)
)

[Parallel(n_jobs=44)]: Using backend LokyBackend with 44 concurrent workers.
[Parallel(n_jobs=44)]: Done   5 out of  50 | elapsed:  4.9min remaining: 44.2min
[Parallel(n_jobs=44)]: Done  11 out of  50 | elapsed:  4.9min remaining: 17.5min
[Parallel(n_jobs=44)]: Done  17 out of  50 | elapsed:  5.0min remaining:  9.6min
[Parallel(n_jobs=44)]: Done  23 out of  50 | elapsed:  5.0min remaining:  5.8min
[Parallel(n_jobs=44)]: Done  29 out of  50 | elapsed:  5.0min remaining:  3.6min
[Parallel(n_jobs=44)]: Done  35 out of  50 | elapsed:  5.0min remaining:  2.1min
[Parallel(n_jobs=44)]: Done  41 out of  50 | elapsed:  5.0min remaining:  1.1min
[Parallel(n_jobs=44)]: Done  47 out of  50 | elapsed:  7.7min remaining:   29.4s
[Parallel(n_jobs=44)]: Done  50 out of  50 | elapsed:  7.7min finished


In [18]:
results.sort(key=lambda d: d["run"])

# 9°: Adding the constant energies

In [19]:
for run_data in results:

    raw_final_energy = run_data["final_energy"]
    true_final_energy = float(interpret_exp_val(raw_final_energy, reduced_molecule_problem))

    run_data["final_energy"] = true_final_energy

    raw_trajectory = run_data["energies_trajectory"]

    true_trajectory = [
        interpret_exp_val(step_energy, reduced_molecule_problem)
        for step_energy in raw_trajectory
    ]

    run_data["energies_trajectory"] = true_trajectory

# 10º Save the data for later analysis.

In [20]:
df = pd.DataFrame(results)

df["optimal_params_json"] = df["optimal_params"].apply(json.dumps)
df["energies_trajectory_json"] = df["energies_trajectory"].apply(json.dumps)
df["params_trajectory_json"] = df["params_trajectory"].apply(json.dumps)

df.to_csv(
    "BEH2_VQE_QPSO_NOISE_FREE.csv",
    columns=[
        "optimizer",
        "run",
        "qpso_seed",
        "steps_requested",
        "steps_done",
        "cost_function_evaluation",
        "energies_trajectory_len",
        "execution_time",
        "final_energy",
        "optimal_params_json",
        "energies_trajectory_json",
        "params_trajectory_json",
    ],
    index=False,
)

print("\nSaved: BEH2_VQE_QPSO_NOISE_FREE.csv")
print("Done.")


Saved: BEH2_VQE_QPSO_NOISE_FREE.csv
Done.
